
## Kovács Levente, Tasnádi Bálint

## Objectives

In our machine learning project, we aim to develop a convulational neural network (CNN) based system that can be used to detect pathologic malformations (in particular, signs of Invasive Ductal Carcinoma) in breast tissue based on hystology slides.

We plan to use the following dataset: https://www.kaggle.com/datasets/paultimothymooney/breast-histopathology-images Introduction to Machine Learning project: Breast cancer classificationthology-images

Our plan is to build a classifier by using approximately 80% of our posessed data. Furthermore, we intend to create a confusion matrix, so we can evaluate our model's performance.

## Subject overview

IDC is the most frequent form of breast cancer. In order to form a prognosis, the agressiveness of the cancer must be evaluated. This is done by human visual inspection of histopathology slides, where the (small) cancerous regions are identified in the large benign background.

ML approaches before CNN (SVM, PCA, autoencoders, random forest) relied on hand-crafted feature detection (colour, edges, density, ...), and achieved accuracies around 80%.

## Reference
Data and context from:


Angel Cruz-Roa, Ajay Basavanhally, Fabio González, et al. "Automatic detection of invasive ductal carcinoma in whole slide images with convolutional neural networks", Proc. SPIE 9041, Medical Imaging 2014: Digital Pathology, 904103 (20 Mar 2014); https://doi.org/10.1117/12.2043872

In [10]:
from glob import glob #filenames
from os import path

import cv2 #image processing

import matplotlib.pyplot as plt

import pandas as pd #data storage
import numpy as np
import kagglehub
import random

In [11]:
# Download latest version
path = kagglehub.dataset_download("paultimothymooney/breast-histopathology-images")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'breast-histopathology-images' dataset.
Path to dataset files: /kaggle/input/breast-histopathology-images


## Dataset description

Humongous (3GB) dataset from 162 breast cancer specimens, broken into 277,524 50x50 patches (198,738 negative, 78,786 positive).

In [12]:
data_filenames = glob(path + '/IDC_regular_ps50_idx5/**/**/*.png') #finds all .png files in data/ and subdirectories

print(f'Found {len(data_filenames)} files')

Found 277524 files


for playing around, let's select 10k positive and 10k negative images. There is a 1:2 class imbalance in the original dataset, we can circumvent it by only working on a small subset for now.

In [13]:
import re

# Assuming `data_filenames` is available from previous cells.
# If not, ensure it's generated, e.g.:
# data_filenames = glob(path + '/IDC_regular_ps50_idx5/**/**/*.png')

# Create a dictionary to map patient IDs to their filepaths, categorized by class
patient_filepaths_categorized_original = {}
for filepath in data_filenames:
    match = re.search(r'IDC_regular_ps50_idx5/(\d+)/', filepath)
    if match:
        patient_id = match.group(1)
        if patient_id not in patient_filepaths_categorized_original:
            patient_filepaths_categorized_original[patient_id] = {'class0': [], 'class1': []}

        if '_class1.png' in filepath:
            patient_filepaths_categorized_original[patient_id]['class1'].append(filepath)
        else:
            patient_filepaths_categorized_original[patient_id]['class0'].append(filepath)

total_positive_mosaics_original = 0
total_negative_mosaics_original = 0

for pid, categories in patient_filepaths_categorized_original.items():
    total_positive_mosaics_original += len(categories['class1'])
    total_negative_mosaics_original += len(categories['class0'])

print(f"Total positive mosaics before undersampling: {total_positive_mosaics_original}")
print(f"Total negative mosaics before undersampling: {total_negative_mosaics_original}")

Total positive mosaics before undersampling: 78786
Total negative mosaics before undersampling: 198738


In [14]:
positive_filepaths = [f for f in data_filenames if f.endswith('_class1.png')]
negative_filepaths = [f for f in data_filenames if f.endswith('_class0.png')]

num_samples = 10000

selected_positive_filepaths = random.sample(positive_filepaths, min(num_samples, len(positive_filepaths)))
selected_negative_filepaths = random.sample(negative_filepaths, min(num_samples, len(negative_filepaths)))

print(f'Selected {len(selected_positive_filepaths)} positive images and {len(selected_negative_filepaths)} negative images.')

# Combine and shuffle the selected file paths
selected_filepaths = selected_positive_filepaths + selected_negative_filepaths
random.shuffle(selected_filepaths)

Selected 10000 positive images and 10000 negative images.


data processing: save into image and label arrays

In [15]:
def load_and_preprocess_image(filepath, target_size=(50, 50)):
    img = cv2.imread(filepath) #read image at path
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB

    #check if 50x50x3
    if img.shape[0] != target_size[0] or img.shape[1] != target_size[1]:
        return None # Exclude images that are not 50x50
    return img


images = []
labels = []

for filepath in selected_filepaths:
    img = load_and_preprocess_image(filepath)
    if img is not None:
        images.append(img)
        # Extract label from filename (class0 or class1)
        label = int(filepath.split('_class')[-1].split('.')[0])
        labels.append(label)

data_20k_images = np.array(images)
data_20k_labels = np.array(labels)

print(f'Shape of loaded images array: {data_20k_images.shape}')
print(f'Shape of loaded labels array: {data_20k_labels.shape}')

Shape of loaded images array: (19898, 50, 50, 3)
Shape of loaded labels array: (19898,)


In [ ]:
np.save('data_20k_images.npy', data_20k_images)

## Example images

In [ ]:
if len(data_20k_images) > 0:
    plt.imshow(data_20k_images[0])
else:
    print("data_20k_images is empty. Please check the data loading steps in earlier cells, especially cell H79PUecxZrLO.")

In [ ]:
positive_images_display = []
negative_images_display = []

positive_count = 0
negative_count = 0

for i in range(len(data_20k_images)):
    if data_20k_labels[i] == 1 and positive_count < 5: #if positive save to positive
        positive_images_display.append(data_20k_images[i])
        positive_count += 1
    elif data_20k_labels[i] == 0 and negative_count < 5: #if negative save to negative
        negative_images_display.append(data_20k_images[i])
        negative_count += 1

    if positive_count == 5 and negative_count == 5: #break if done
        break

plt.figure(figsize=(12, 5))

#positive
for i in range(5):
    plt.subplot(2, 5, i + 1)
    plt.imshow(positive_images_display[i])
    plt.axis('off')
    plt.title("positive")

#negative
for i in range(5):
    plt.subplot(2, 5, i + 6)
    plt.imshow(negative_images_display[i])
    plt.title("negative")
    plt.axis('off')

### Complete slide

In [ ]:
import re

# Define the base directory for the specific slide
slide_base_dir = '/kaggle/input/breast-histopathology-images/IDC_regular_ps50_idx5/10253'

# Find all .png files within the 0 and 1 subfolders of the specified slide
all_slide_filepaths = glob(f'{slide_base_dir}/**/*.png', recursive=True)

print(f'Found {len(all_slide_filepaths)} image patches for slide 10253.')

In [ ]:
def load_mosaic_image_with_coords(filepath):
    img = cv2.imread(filepath) # Read image
    if img is None:
        return None, None, None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB

    # Extract x and y coordinates from the filename
    match = re.search(r'_x(\d+)_y(\d+)_', filepath)
    if match:
        x = int(match.group(1))
        y = int(match.group(2))
        return img, x, y
    return None, None, None

# Assuming each mosaic image is 50x50 pixels
MOSAIC_SIZE = 50

In [ ]:
mosaic_images = []
coords = []

min_x, max_x = float('inf'), float('-inf')
min_y, max_y = float('inf'), float('-inf')

for filepath in all_slide_filepaths:
    img, x, y = load_mosaic_image_with_coords(filepath)
    if img is not None:
        mosaic_images.append(img)
        coords.append((x, y))
        min_x = min(min_x, x)
        max_x = max(max_x, x)
        min_y = min(min_y, y)
        max_y = max(max_y, y)

print(f'Processed {len(mosaic_images)} mosaic images.')

# Calculate the dimensions of the full slide
# The width will be (max_x - min_x + MOSAIC_SIZE) and height (max_y - min_y + MOSAIC_SIZE)
# Since coordinates are top-left corners, add MOSAIC_SIZE to get the full span
full_slide_width = max_x - min_x + MOSAIC_SIZE
full_slide_height = max_y - min_y + MOSAIC_SIZE

print(f'Full slide dimensions: {full_slide_width}x{full_slide_height} pixels.')

# Create a blank canvas for the full slide
full_slide = np.zeros((full_slide_height, full_slide_width, 3), dtype=np.uint8)

In [ ]:
for i, (img, (x, y)) in enumerate(zip(mosaic_images, coords)):
    # Calculate relative position on the canvas
    rel_x = x - min_x
    rel_y = y - min_y
    full_slide[rel_y : rel_y + MOSAIC_SIZE, rel_x : rel_x + MOSAIC_SIZE] = img

# Display the complete slide
plt.figure(figsize=(15, 15))
plt.imshow(full_slide)
plt.title(f'Complete Slide Reconstruction for 10253 ({full_slide_width}x{full_slide_height})')
plt.axis('off')
plt.show()

In [ ]:
# Re-create a blank canvas for the full slide, this time for tinted images
full_slide_tinted = np.zeros((full_slide_height, full_slide_width, 3), dtype=np.uint8)

for filepath in all_slide_filepaths:
    img, x, y = load_mosaic_image_with_coords(filepath)
    if img is not None:
        img_to_place = img.copy() # Make a copy to avoid modifying original loaded images if they were stored

        # Check if the image is from class1 (cancerous)
        if '_class1.png' in filepath:
            # Apply a red tint: boost red channel, slightly reduce green and blue
            img_to_place[:, :, 0] = np.clip(img_to_place[:, :, 0] * 1.2, 0, 255) # Red channel (boost)
            img_to_place[:, :, 1] = np.clip(img_to_place[:, :, 1] * 0.8, 0, 255) # Green channel (reduce)
            img_to_place[:, :, 2] = np.clip(img_to_place[:, :, 2] * 0.8, 0, 255) # Blue channel (reduce)

        # Calculate relative position on the canvas
        rel_x = x - min_x
        rel_y = y - min_y
        full_slide_tinted[rel_y : rel_y + MOSAIC_SIZE, rel_x : rel_x + MOSAIC_SIZE] = img_to_place

# Display the complete slide with tinted class1 images
plt.figure(figsize=(15, 15))
plt.imshow(full_slide_tinted)
plt.title(f'Complete Slide Reconstruction with Tinted Class 1 ({full_slide_width}x{full_slide_height})')
plt.axis('off')
plt.show()

In [ ]:
if history is not None:
    # Find the epoch with the minimum validation loss
    best_epoch_index = np.argmin(history.history['val_loss'])
    best_val_loss = history.history['val_loss'][best_epoch_index]
    best_val_accuracy = history.history['val_accuracy'][best_epoch_index]


    print(f"EarlyStopping reverted to epoch {best_epoch_index} with the best validation loss:")
    print(f"  Validation Loss: {best_val_loss:.4f}")
    print(f"  Validation Accuracy: {best_val_accuracy:.4f}")
else:
    print("History object not found. Please ensure the model was trained with 'history = model.fit(...)'")

The red tinted regions were manually identified to be cancerous by a pathologist. Our goal is to get a similar identification using a CNN.

# 20k set

## Train, test, validate sets


We separated the data, the ratio was the following: 80% for training, 10% for validation and 10% testing. The separation was done with the help of sklearn library.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X = data_20k_images
y = data_20k_labels

X = X.astype('float32') / 255.0

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(X_train.shape[0])
print(X_val.shape[0])
print(X_test.shape[0])

The CNN model was built. We used tensorflow because it is a high level API which means it is easier for us to build and handle the network. In case we used PyTorch the process would be slightly complicated. Moreover, tensorflow was created by Google so as Colab, which is our running envinronment this time, overall this makes a perfect match.

## Model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(50, 50, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Recall()])

model.summary()
print("Success!")

## Training:

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping #helps prevent overfitting

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

In [ ]:
model.save('breast_cancer_model.keras')
print("Success!")

In [ ]:
# Plotting the Training and Validation Accuracy & Loss
plt.figure(figsize=(12, 5))

# Accuracy Graph
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='lower right')

# Loss Graph
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

## Confusion matrix:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

y_pred_prob = model.predict(X_test)

y_pred = (y_pred_prob > 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Healthy (0)', 'Cancerous (1)'],
            yticklabels=['Healthy (0)', 'Cancerous (1)'],
            annot_kws={"size": 16})

plt.title('Confusion Matrix - Test Set', fontsize=16)
plt.ylabel('True Label', fontsize=14)
plt.xlabel('Predicted Label', fontsize=14)
plt.show()

print(classification_report(y_test, y_pred, target_names=['Healthy (0)', 'Cancerous (1)']))

In [ ]:
tn, fp, fn, tp = cm.ravel()

# False Positive Percentage: (False Positives / All Actual Negatives)
# All Actual Negatives = True Negatives + False Positives
false_positive_percentage = (fp / (fp + tn)) * 100

# False Negative Percentage: (False Negatives / All Actual Positives)
# All Actual Positives = True Positives + False Negatives
false_negative_percentage = (fn / (fn + tp)) * 100

print(f"False Positive Percentage: {false_positive_percentage:.2f}%")
print(f"False Negative Percentage: {false_negative_percentage:.2f}%")

# Large dataset with undersampling
Training on a larger database, in this case we used as much data as we could to complete the training, however we held the balance between the number of healthy and cancerous training pictures. This method is called undersampling.



## Train, test, validate sets
Now we pay attention to break up our dataset along patient IDs:

In [ ]:
import re
from sklearn.model_selection import train_test_split
import random

# Assuming `data_filenames` from the previous cells contains all filepaths
# If not, regenerate it based on the path variable
# data_filenames = glob(path + '/IDC_regular_ps50_idx5/**/**/*.png')

# Create a dictionary to map patient IDs to their filepaths, categorized by class
patient_filepaths_categorized = {}
for filepath in data_filenames:
    match = re.search(r'IDC_regular_ps50_idx5/(\d+)/', filepath)
    if match:
        patient_id = match.group(1)
        if patient_id not in patient_filepaths_categorized:
            patient_filepaths_categorized[patient_id] = {'class0': [], 'class1': []}

        if '_class1.png' in filepath:
            patient_filepaths_categorized[patient_id]['class1'].append(filepath)
        else:
            patient_filepaths_categorized[patient_id]['class0'].append(filepath)

unique_patient_ids = list(patient_filepaths_categorized.keys())
print(f'Total unique patient IDs found: {len(unique_patient_ids)}')

# Implement balanced undersampling per patient (equal positives and negatives)
patient_filepaths_undersampled = {}
for pid, categories in patient_filepaths_categorized.items():
    original_positive_mosaics = categories['class1']
    original_negative_mosaics = categories['class0']

    # Determine the number of mosaics to keep for each class to ensure balance
    num_mosaics_to_keep_per_class = min(len(original_positive_mosaics), len(original_negative_mosaics))

    sampled_positive_mosaics = []
    sampled_negative_mosaics = []

    if num_mosaics_to_keep_per_class > 0:
        # Randomly sample positive mosaics
        sampled_positive_mosaics = random.sample(original_positive_mosaics, num_mosaics_to_keep_per_class)
        # Randomly sample negative mosaics
        sampled_negative_mosaics = random.sample(original_negative_mosaics, num_mosaics_to_keep_per_class)

    # Combine the sampled mosaics
    patient_filepaths_undersampled[pid] = sampled_positive_mosaics + sampled_negative_mosaics
    random.shuffle(patient_filepaths_undersampled[pid]) # Shuffle to mix classes

# Filter out patients with no mosaics after undersampling (e.g., if a patient only had negatives and no positives)
filtered_patient_ids = [pid for pid, filepaths in patient_filepaths_undersampled.items() if filepaths]

# Split filtered patient IDs into train, validation, and test sets (80-10-10)
patient_train_ids, patient_temp_ids = train_test_split(
    filtered_patient_ids, test_size=0.2, random_state=42
)
patient_val_ids, patient_test_ids = train_test_split(
    patient_temp_ids, test_size=0.5, random_state=42
)

print(f'\nNumber of patients in training set: {len(patient_train_ids)}')
print(f'Number of patients in validation set: {len(patient_val_ids)}')
print(f'Number of patients in test set: {len(patient_test_ids)}')

# Function to count mosaics for a given list of patient IDs
def count_mosaics(patient_ids, all_patient_filepaths_map):
    total_mosaics = 0
    total_positives = 0
    total_negatives = 0
    for pid in patient_ids:
        filepaths = all_patient_filepaths_map.get(pid, [])
        total_mosaics += len(filepaths)
        for filepath in filepaths:
            if '_class1.png' in filepath:
                total_positives += 1
            else:
                total_negatives += 1
    return total_mosaics, total_positives, total_negatives

# Count mosaics for each set using the undersampled data
train_mosaics, train_pos, train_neg = count_mosaics(patient_train_ids, patient_filepaths_undersampled)
val_mosaics, val_pos, val_neg = count_mosaics(patient_val_ids, patient_filepaths_undersampled)
test_mosaics, test_pos, test_neg = count_mosaics(patient_test_ids, patient_filepaths_undersampled)

print(f'\nTotal mosaics in training set: {train_mosaics} (Positive: {train_pos}, Negative: {train_neg})')
print(f'Total mosaics in validation set: {val_mosaics} (Positive: {val_pos}, Negative: {val_neg})')
print(f'Total mosaics in test set: {test_mosaics} (Positive: {test_pos}, Negative: {test_neg})')

### Original class distribution
With this undersampling, the positives end up being slightly oversampled:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Prepare data for plotting
patient_mosaic_counts = []
for pid, categories in patient_filepaths_categorized_original.items():
    patient_mosaic_counts.append({
        'patient_id': pid,
        'positive_mosaics': len(categories['class1']),
        'negative_mosaics': len(categories['class0'])
    })

df_patient_counts = pd.DataFrame(patient_mosaic_counts)

# Sort by patient_id for better visualization
df_patient_counts['patient_id_int'] = df_patient_counts['patient_id'].astype(int)
df_patient_counts = df_patient_counts.sort_values('patient_id_int').drop(columns='patient_id_int')

# Create the plot
plt.figure(figsize=(20, 10)) # Adjust figure size for readability

plt.bar(df_patient_counts['patient_id'], df_patient_counts['positive_mosaics'], color='red', label='Positive Mosaics')
plt.bar(df_patient_counts['patient_id'], df_patient_counts['negative_mosaics'], bottom=df_patient_counts['positive_mosaics'], color='blue', label='Negative Mosaics')

plt.xlabel('Patient ID', fontsize=12)
plt.ylabel('Number of Mosaics', fontsize=12)
plt.title('Number of Positive and Negative Mosaics per Patient (Original Dataset)', fontsize=14)
plt.xticks(rotation=90, fontsize=8) # Rotate x-axis labels for better readability
plt.yticks(fontsize=10)
plt.legend(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()

In [ ]:
from tqdm.notebook import tqdm # Import tqdm for progress bar

# Function to load images and labels for a given list of patient IDs
def load_data_for_patients(patient_ids, all_patient_filepaths_map, target_size=(50, 50)):
    images = []
    labels = []
    # Wrap the patient_ids iteration with tqdm for a progress bar
    for pid in tqdm(patient_ids, desc="Loading images for patients"):
        filepaths = all_patient_filepaths_map.get(pid, [])
        for filepath in filepaths:
            img = load_and_preprocess_image(filepath, target_size)
            if img is not None:
                images.append(img)
                label = int(filepath.split('_class')[-1].split('.')[0])
                labels.append(label)
    return np.array(images), np.array(labels)

# Load data for training set
X_train_patient, y_train_patient = load_data_for_patients(patient_train_ids, patient_filepaths_undersampled)
print(f"X_train_patient shape: {X_train_patient.shape}, y_train_patient shape: {y_train_patient.shape}")

# Load data for validation set
X_val_patient, y_val_patient = load_data_for_patients(patient_val_ids, patient_filepaths_undersampled)
print(f"X_val_patient shape: {X_val_patient.shape}, y_val_patient shape: {y_val_patient.shape}")

# Load data for test set
X_test_patient, y_test_patient = load_data_for_patients(patient_test_ids, patient_filepaths_undersampled)
print(f"X_test_patient shape: {X_test_patient.shape}, y_test_patient shape: {y_test_patient.shape}")

In [ ]:
# Save the patient-wise split data
np.save('X_train_patient.npy', X_train_patient)
np.save('y_train_patient.npy', y_train_patient)
np.save('X_val_patient.npy', X_val_patient)
np.save('y_val_patient.npy', y_val_patient)
np.save('X_test_patient.npy', X_test_patient)
np.save('y_test_patient.npy', y_test_patient)

print("Patient-wise split data saved successfully.")

## Model

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Redefine the CNN model (as it might have been trained on previous data)
model_patient_wise = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(50, 50, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_patient_wise.compile(optimizer='adam',
                           loss='binary_crossentropy',
                           metrics=['accuracy', tf.keras.metrics.Recall()])

model_patient_wise.summary()



## Training
Now, we will redefine the model and train it using the patient-wise split data, incorporating class weights and early stopping.

In [ ]:
# Setup EarlyStopping
early_stop_patient = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history_patient_wise = model_patient_wise.fit(
    X_train_patient,
    y_train_patient,
    epochs=30,
    batch_size=64, # Use a batch size here, or define it explicitly
    validation_data=(X_val_patient, y_val_patient),
    callbacks=[early_stop_patient]
)

## History

In [ ]:
# Plotting the Training and Validation Accuracy & Loss
plt.figure(figsize=(12, 5))

# Accuracy Graph
plt.subplot(1, 2, 1)
plt.plot(history_patient_wise.history['accuracy'], label='Train Accuracy')
plt.plot(history_patient_wise.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy (Patient-wise)')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='lower right')

# Loss Graph
plt.subplot(1, 2, 2)
plt.plot(history_patient_wise.history['loss'], label='Train Loss')
plt.plot(history_patient_wise.history['val_loss'], label='Validation Loss')
plt.title('Model Loss (Patient-wise)')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')


plt.tight_layout()
plt.show()

## Confusion matrix

In [ ]:

# Evaluate the model on the test set
y_pred_prob_patient = model_patient_wise.predict(X_test_patient)
y_pred_patient = (y_pred_prob_patient > 0.5).astype(int)

# Confusion Matrix
cm_patient = confusion_matrix(y_test_patient, y_pred_patient)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_patient, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Healthy (0)', 'Cancerous (1)'],
            yticklabels=['Healthy (0)', 'Cancerous (1)'],
            annot_kws={"size": 16})
plt.title('Confusion Matrix - Patient-wise Test Set', fontsize=16)
plt.ylabel('True Label', fontsize=14)
plt.xlabel('Predicted Label', fontsize=14)
plt.show()

print("\n--- CLASSIFICATION REPORT (Patient-wise Test Set) ---")
print(classification_report(y_test_patient, y_pred_patient, target_names=['Healthy (0)', 'Cancerous (1)']))

In [ ]:
tn, fp, fn, tp = cm_patient.ravel()

# False Positive Percentage: (False Positives / All Actual Negatives)
# All Actual Negatives = True Negatives + False Positives
false_positive_percentage = (fp / (fp + tn)) * 100

# False Negative Percentage: (False Negatives / All Actual Positives)
# All Actual Positives = True Positives + False Negatives
false_negative_percentage = (fn / (fn + tp)) * 100

print(f"False Positive Percentage: {false_positive_percentage:.2f}%")
print(f"False Negative Percentage: {false_negative_percentage:.2f}%")

## Complete slide predictions

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import re
import cv2 # Ensure cv2 is imported
from sklearn.metrics import confusion_matrix, classification_report

# Assuming each mosaic image is 50x50 pixels
MOSAIC_SIZE = 50

# Helper function to load mosaic image and extract coordinates
def load_mosaic_image_with_coords(filepath, target_size=(MOSAIC_SIZE, MOSAIC_SIZE)):
    try:
        img = cv2.imread(filepath) # Read image
        if img is None:
            return None, None, None
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB

        # Check for correct number of dimensions and shape (height, width, channels)
        if not (len(img.shape) == 3 and img.shape[0] == target_size[0] and \
                img.shape[1] == target_size[1] and img.shape[2] == 3):
            return None, None, None # Exclude images that are not (50, 50, 3)

        # Extract x and y coordinates from the filename
        match = re.search(r'_x(\d+)_y(\d+)_', filepath)
        if match:
            x = int(match.group(1))
            y = int(match.group(2))
            return img, x, y
        return None, None, None
    except Exception as e:
        print(f"Error processing image {filepath}: {e}")
        return None, None, None

# Helper function to apply specific tints
def apply_color_tint(image, tint_type):
    tinted_image = image.copy().astype(np.float32)
    if tint_type == 'green': # True Positive (Actual 1, Predicted 1)
        tinted_image[:, :, 1] = np.clip(tinted_image[:, :, 1] * 1.5, 0, 255) # Boost green
        tinted_image[:, :, 0] = np.clip(tinted_image[:, :, 0] * 0.8, 0, 255) # Reduce red
        tinted_image[:, :, 2] = np.clip(tinted_image[:, :, 2] * 0.8, 0, 255) # Reduce blue
    elif tint_type == 'yellow': # False Positive (Actual 0, Predicted 1)
        tinted_image[:, :, 0] = np.clip(tinted_image[:, :, 0] * 1.2, 0, 255) # Boost red
        tinted_image[:, :, 1] = np.clip(tinted_image[:, :, 1] * 1.2, 0, 255) # Boost green
        tinted_image[:, :, 2] = np.clip(tinted_image[:, :, 2] * 0.7, 0, 255) # Reduce blue
    elif tint_type == 'blue': # False Negative (Actual 1, Predicted 0)
        tinted_image[:, :, 2] = np.clip(tinted_image[:, :, 2] * 1.5, 0, 255) # Boost blue
        tinted_image[:, :, 0] = np.clip(tinted_image[:, :, 0] * 0.8, 0, 255) # Reduce red
        tinted_image[:, :, 1] = np.clip(tinted_image[:, :, 1] * 0.8, 0, 255) # Reduce green
    elif tint_type == 'desaturated': # True Negative (Actual 0, Predicted 0)
        # Desaturate slightly to make other colors stand out
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        tinted_image = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB).astype(np.float32)
        tinted_image = tinted_image * 0.8 + image * 0.2 # Mix with original slightly for texture
    return tinted_image.astype(np.uint8)


num_plots_to_generate = 5
plots_generated_count = 0

print(f"Generating visualizations for {num_plots_to_generate} test patients using all their original mosaics:")

# Limit the loop to only the first `num_plots_to_generate` patients
for patient_id in patient_test_ids[:num_plots_to_generate]:
    print(f"Processing patient ID: {patient_id}")

    # Gather all filepaths for this patient from the ORIGINAL patient-wise split data
    patient_original_filepaths_dict = patient_filepaths_categorized_original.get(patient_id, {'class0': [], 'class1': []})
    all_filepaths_for_patient = patient_original_filepaths_dict['class0'] + patient_original_filepaths_dict['class1']

    if not all_filepaths_for_patient:
        print(f"No mosaics found for patient {patient_id}. Skipping.")
        continue

    patient_mosaics_raw = []
    patient_mosaics_preprocessed = []
    patient_mosaic_coords = []
    patient_true_labels = []

    min_x, max_x = float('inf'), float('-inf')
    min_y, max_y = float('inf'), float('-inf')

    for filepath in all_filepaths_for_patient:
        img_raw, x, y = load_mosaic_image_with_coords(filepath)
        if img_raw is not None:
            patient_mosaics_raw.append(img_raw)
            # FIX: Remove scaling here. Model was trained on 0-255 range.
            patient_mosaics_preprocessed.append(img_raw.astype('float32')) # No division by 255.0
            patient_mosaic_coords.append((x, y))
            patient_true_labels.append(1 if '_class1.png' in filepath else 0)

            # Update min/max coordinates
            min_x = min(min_x, x)
            max_x = max(max_x, x)
            min_y = min(min_y, y)
            max_y = max(max_y, y)

    if not patient_mosaics_preprocessed:
        print(f"No valid mosaics loaded for patient {patient_id}. Skipping.")
        continue

    patient_mosaics_array = np.array(patient_mosaics_preprocessed)

    # Make predictions using the patient-wise trained model
    predictions_prob = model_patient_wise.predict(patient_mosaics_array, verbose=0) # suppress verbose output
    predictions_binary = (predictions_prob > 0.5).astype(int).flatten()

    # Removed: all_mosaic_true_labels.extend(patient_true_labels)
    # Removed: all_mosaic_predicted_labels.extend(predictions_binary)

    # --- Debug print for prediction probabilities for the current patient ---
    print(f"  Prediction Probabilities for Patient {patient_id}:")
    print(f"    Min: {predictions_prob.min():.4f}, Max: {predictions_prob.max():.4f}")
    print(f"    Mean: {predictions_prob.mean():.4f}, Std: {predictions_prob.std():.4f}")

    # Calculate per-patient predicted positive/negative counts
    predicted_positives_patient = np.sum(predictions_binary == 1)
    predicted_negatives_patient = np.sum(predictions_binary == 0)
    print(f"  Predicted Positives for Patient {patient_id}: {predicted_positives_patient}")
    print(f"  Predicted Negatives for Patient {patient_id}: {predicted_negatives_patient}")
    # -----------------------------------------------------------------------

    # Only generate plots for the first `num_plots_to_generate` patients
    if plots_generated_count < num_plots_to_generate:
        # Calculate full slide dimensions
        full_slide_width = max_x - min_x + MOSAIC_SIZE
        full_slide_height = max_y - min_y + MOSAIC_SIZE

        # Create a blank canvas for the reconstructed slide (tinted based on true/pred)
        reconstructed_slide = np.zeros((full_slide_height, full_slide_width, 3), dtype=np.uint8)
        # Create a blank canvas for the probability heatmap
        probability_heatmap = np.zeros((full_slide_height, full_slide_width), dtype=np.float32)

        # Populate the canvas with mosaics, applying tints based on true vs predicted labels
        for i in range(len(patient_mosaics_raw)):
            img_raw = patient_mosaics_raw[i]
            x, y = patient_mosaic_coords[i]
            true_label = patient_true_labels[i]
            predicted_label = predictions_binary[i]
            prediction_score = predictions_prob[i][0] # Get the single probability score

            rel_x = x - min_x
            rel_y = y - min_y

            tint_type = 'original' # Default, though it will be overwritten
            if true_label == 1 and predicted_label == 1:
                tint_type = 'green'  # True Positive
            elif true_label == 0 and predicted_label == 1:
                tint_type = 'yellow' # False Positive
            elif true_label == 1 and predicted_label == 0:
                tint_type = 'blue'   # False Negative
            else: # true_label == 0 and predicted_label == 0
                tint_type = 'desaturated' # True Negative

            tinted_img = apply_color_tint(img_raw, tint_type)
            reconstructed_slide[rel_y : rel_y + MOSAIC_SIZE, rel_x : rel_x + MOSAIC_SIZE] = tinted_img

            # Fill the probability heatmap
            probability_heatmap[rel_y : rel_y + MOSAIC_SIZE, rel_x : rel_x + MOSAIC_SIZE] = prediction_score

        # Display the reconstructed slide and the heatmap side by side
        fig, axes = plt.subplots(1, 2, figsize=(30, 18)) # Increased width for two plots

        # Plot 1: Reconstructed Slide with Prediction Categories
        axes[0].imshow(reconstructed_slide)
        axes[0].set_title(f'Patient ID: {patient_id} - Predictions by Category', fontsize=16)
        axes[0].axis('off')

        # Plot 2: Probability Heatmap
        im = axes[1].imshow(probability_heatmap, cmap='viridis', vmin=0, vmax=1) # Use viridis or similar colormap
        axes[1].set_title(f'Patient ID: {patient_id} - Prediction Probability Heatmap', fontsize=16)
        axes[1].axis('off')

        # Add a colorbar for the heatmap
        cbar = fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        cbar.set_label('Prediction Probability (0=Healthy, 1=Cancerous)', fontsize=12)

        # Create a single legend manually on the figure below the subplots
        # Adjusting y-coordinates and using fig.text for a global legend
        fig.text(0.02, 0.01, 'Green: True Positive (Actual 1, Pred 1)', color='green', fontsize=10, transform=fig.transFigure)
        fig.text(0.02, -0.01, 'Yellow: False Positive (Actual 0, Pred 1)', color='orange', fontsize=10, transform=fig.transFigure)
        fig.text(0.02, -0.03, 'Blue: False Negative (Actual 1, Pred 0)', color='blue', fontsize=10, transform=fig.transFigure)
        fig.text(0.02, -0.05, 'Desaturated: True Negative (Actual 0, Pred 0)', color='gray', fontsize=10, transform=fig.transFigure)

        plt.tight_layout(rect=[0, 0.06, 1, 1]) # Adjust layout to prevent legend from being cut off
        plt.show()

        plots_generated_count += 1

Training on balnced dataset (as big as possible)

In [ ]:
import pandas as pd
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
import kagglehub

In [16]:
# Download dataset
path = kagglehub.dataset_download("paultimothymooney/breast-histopathology-images")
all_filepaths = glob(path + '/IDC_regular_ps50_idx5/**/*.png', recursive=True)

# Extract labels and patient IDs
labels = ['1' if f.endswith('_class1.png') else '0' for f in all_filepaths]
patient_ids = [f.split('/')[-3] for f in all_filepaths]

# Create DataFrame
df = pd.DataFrame({'filepath': all_filepaths, 'label': labels, 'patient_id': patient_ids})

# Patient-level split to prevent data leakage (80% Train, 10% Val, 10% Test)
unique_patients = df['patient_id'].unique()
train_patients, temp_patients = train_test_split(unique_patients, test_size=0.20, random_state=42)
val_patients, test_patients = train_test_split(temp_patients, test_size=0.50, random_state=42)

train_df = df[df['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df = df[df['patient_id'].isin(val_patients)].reset_index(drop=True)
test_df = df[df['patient_id'].isin(test_patients)].reset_index(drop=True)

print("--- DATASET SPLIT INFO ---")
print(f"Total training patches: {len(train_df)}")
print(f"Total validation patches: {len(val_df)}")
print(f"Total test patches: {len(test_df)}")

Loading 10000 images...
Starting training...
Epoch 1/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7031 - loss: 0.5669 - val_accuracy: 0.7950 - val_loss: 0.4604
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7742 - loss: 0.4837 - val_accuracy: 0.7780 - val_loss: 0.4790
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7891 - loss: 0.4633 - val_accuracy: 0.7120 - val_loss: 0.5721
Epoch 4/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7905 - loss: 0.4586 - val_accuracy: 0.8170 - val_loss: 0.4421
Epoch 5/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8002 - loss: 0.4406 - val_accuracy: 0.7750 - val_loss: 0.5042
Epoch 6/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8005 - loss: 0.4281 - val_accuracy: 0.8100 - val_loss: 0.4341
Epoch 7/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8174 - loss: 0.4124 - val_accuracy: 0.8020 - val_loss: 0.4525
Epoch 8/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accur

In [ ]:
# Separate healthy and cancer patches in the training set
df_healthy_train = train_df[train_df['label'] == '0']
df_cancer_train = train_df[train_df['label'] == '1']

# Find the maximum possible balanced size (the total size of the minority class)
min_size_train = min(len(df_healthy_train), len(df_cancer_train))

# Perform undersampling
train_df_balanced = pd.concat([
    df_healthy_train.sample(n=min_size_train, random_state=42),
    df_cancer_train.sample(n=min_size_train, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print("\n--- UNDERSAMPLING INFO ---")
print(f"Balanced training set size: {len(train_df_balanced)} patches")
print(f"Healthy patches (Class 0): {min_size_train}")
print(f"Cancerous patches (Class 1): {min_size_train}")

# Create Data Generators
batch_size = 64
datagen = ImageDataGenerator(rescale=1./255)

train_generator_under = datagen.flow_from_dataframe(
    dataframe=train_df_balanced, x_col='filepath', y_col='label',
    target_size=(50, 50), class_mode='binary', batch_size=batch_size, shuffle=True
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df, x_col='filepath', y_col='label',
    target_size=(50, 50), class_mode='binary', batch_size=batch_size, shuffle=False
)

test_generator = datagen.flow_from_dataframe(
    dataframe=test_df, x_col='filepath', y_col='label',
    target_size=(50, 50), class_mode='binary', batch_size=batch_size, shuffle=False
)

# Define the CNN architecture
def create_model():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(50, 50, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Recall(name='recall')])
    return model

In [ ]:
model_under = create_model()

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Starting training with the undersampled (balanced) dataset...")
history_under = model_under.fit(
    train_generator_under,
    epochs=30,
    validation_data=val_generator,
    callbacks=[early_stop]
)
print("Training completed.")

In [ ]:
print("Evaluating model on the Test Set...")
test_generator.reset()
y_pred_prob_under = model_under.predict(test_generator)
y_pred_under = (y_pred_prob_under > 0.5).astype(int)
y_true = test_generator.classes

# Calculate metrics
cm_under = confusion_matrix(y_true, y_pred_under)
tn, fp, fn, tp = cm_under.ravel()

print("\n--- UNDERSAMPLING METRICS ---")
print(f"True Positives (Correctly identified cancer): {tp}")
print(f"True Negatives (Correctly identified healthy): {tn}")
print(f"False Positives (False Alarm / Healthy flagged as cancer): {fp}")
print(f"False Negatives (Missed Cancer / Cancer flagged as healthy): {fn}")

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_true, y_pred_under, target_names=['Healthy (0)', 'Cancerous (1)']))

# Plot Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_under, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Healthy (0)', 'Cancerous (1)'],
            yticklabels=['Healthy (0)', 'Cancerous (1)'],
            annot_kws={"size": 16})

plt.title('Confusion Matrix - Undersampling Model', fontsize=16)
plt.ylabel('True Label', fontsize=14)
plt.xlabel('Predicted Label', fontsize=14)
plt.show()

We conducted the training without balanced dataset out of curiosity and without further change in the model.

In [2]:
print("--- IMBALANCED BASELINE TRAINING ---")
print("Training the model on the full, imbalanced dataset (70% Healthy / 30% Cancerous).")

# We use the original train_df without any undersampling or class weights
train_generator_imb = datagen.flow_from_dataframe(
    dataframe=train_df, x_col='filepath', y_col='label',
    target_size=(50, 50), class_mode='binary', batch_size=batch_size, shuffle=True
)

# Create a fresh, untrained model for this experiment
model_imb = create_model()

# Training Phase
print("\nStarting training with the imbalanced dataset...")
history_imb = model_imb.fit(
    train_generator_imb,
    epochs=30,
    validation_data=val_generator,
    callbacks=[early_stop]
)
print("Training completed.")

# Evaluation Phase
print("\nEvaluating Imbalanced Baseline model on the Test Set...")
test_generator.reset()
y_pred_prob_imb = model_imb.predict(test_generator)
y_pred_imb = (y_pred_prob_imb > 0.5).astype(int)

# The y_true is the same as before (y_true = test_generator.classes)
cm_imb = confusion_matrix(y_true, y_pred_imb)
tn, fp, fn, tp = cm_imb.ravel()

# Metrics and Statistics
print("\n--- IMBALANCED BASELINE METRICS ---")
print(f"True Positives (Correctly identified cancer): {tp}")
print(f"True Negatives (Correctly identified healthy): {tn}")
print(f"False Positives (False Alarm / Healthy flagged as cancer): {fp}")
print(f"False Negatives (Missed Cancer / Cancer flagged as healthy): {fn}")

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_true, y_pred_imb, target_names=['Healthy (0)', 'Cancerous (1)']))

# Plot Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_imb, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Healthy (0)', 'Cancerous (1)'],
            yticklabels=['Healthy (0)', 'Cancerous (1)'],
            annot_kws={"size": 16})

plt.title('Confusion Matrix - Imbalanced Baseline Model', fontsize=16)
plt.ylabel('True Label', fontsize=14)
plt.xlabel('Predicted Label', fontsize=14)
plt.show()

Total images: 277524
Training on: 222019 images
Found 222019 validated image filenames belonging to 2 classes.
Found 27752 validated image filenames belonging to 2 classes.
Epoch 1/10
3470/3470 ━━━━━━━━━━━━━━━━━━━━ 194s 55ms/step - accuracy: 0.8314 - loss: 0.3869 - val_accuracy: 0.8435 - val_loss: 0.3574
Epoch 2/10
3470/3470 ━━━━━━━━━━━━━━━━━━━━ 114s 33ms/step - accuracy: 0.8491 - loss: 0.3514 - val_accuracy: 0.8487 - val_loss: 0.3511
Epoch 3/10
3470/3470 ━━━━━━━━━━━━━━━━━━━━ 113s 33ms/step - accuracy: 0.8555 - loss: 0.3384 - val_accuracy: 0.8532 - val_loss: 0.3369
Epoch 4/10
3470/3470 ━━━━━━━━━━━━━━━━━━━━ 114s 33ms/step - accuracy: 0.8590 - loss: 0.3293 - val_accuracy: 0.8554 - val_loss: 0.3374
Epoch 5/10
3470/3470 ━━━━━━━━━━━━━━━━━━━━ 115s 33ms/step - accuracy: 0.8623 - loss: 0.3226 - val_accuracy: 0.8649 - val_loss: 0.3166
Epoch 6/10
3470/3470 ━━━━━━━━━━━━━━━━━━━━ 110s 32ms/step - accuracy: 0.8656 - loss: 0.3163 - val_accuracy: 0.8672 - val_loss: 0.3146
Epoch 7/10
3470/3470 ━━━━━━━━

In our next training we use weights.By introducing class weights, the model sacrificed its deceptively high overall accuracy to become medically much more sensitive and reliable in detecting critical cancerous cases.

In [ ]:
print("--- WEIGHTED TRAINING ---")
print("By introducing class weights, the model sacrifices its deceptively high overall accuracy")
print("to become medically much more sensitive and reliable in detecting critical cancerous cases.\n")

# Calculate mathematical class weights based on the imbalanced training set
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
)
class_weights = {0: class_weights_array[0], 1: class_weights_array[1]}

print(f"Calculated Class Weights:")
print(f"Weight for Healthy (Class 0): {class_weights[0]:.4f}")
print(f"Weight for Cancerous (Class 1): {class_weights[1]:.4f}\n")

# Create a fresh, untrained model
model_weight = create_model()

# Training Phase (Using the imbalanced generator, but applying the weights)
print("Starting weighted training...")
history_weight = model_weight.fit(
    train_generator_imb,
    epochs=30,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=[early_stop]
)
print("Training completed.")

# Evaluation Phase
print("\nEvaluating Weighted model on the Test Set...")
test_generator.reset()
y_pred_prob_weight = model_weight.predict(test_generator)
y_pred_weight = (y_pred_prob_weight > 0.5).astype(int)

# Calculate metrics
cm_weight = confusion_matrix(y_true, y_pred_weight)
tn, fp, fn, tp = cm_weight.ravel()

# Metrics and Statistics
print("\n--- WEIGHTED MODEL METRICS ---")
print(f"True Positives (Correctly identified cancer): {tp}")
print(f"True Negatives (Correctly identified healthy): {tn}")
print(f"False Positives (False Alarm / Healthy flagged as cancer): {fp}")
print(f"False Negatives (Missed Cancer / Cancer flagged as healthy): {fn}")

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_true, y_pred_weight, target_names=['Healthy (0)', 'Cancerous (1)']))

# Plot Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_weight, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Healthy (0)', 'Cancerous (1)'],
            yticklabels=['Healthy (0)', 'Cancerous (1)'],
            annot_kws={"size": 16})

plt.title('Confusion Matrix - Weighted Model', fontsize=16)
plt.ylabel('True Label', fontsize=14)
plt.xlabel('Predicted Label', fontsize=14)
plt.show()

Patient based classification

In [ ]:
print("--- PATIENT-LEVEL EVALUATION (THE ULTIMATE TEST) ---")
print("Aggregating patch-level predictions to diagnose full patients.")
print("Using the best model: Weighted Training Model.\n")

# Combine predictions with patient IDs
test_df_copy = test_df.copy()
test_df_copy['true_patch_label'] = test_df_copy['label'].astype(int)
test_df_copy['pred_patch_label'] = y_pred_weight

# Group by patient ID
patient_results = test_df_copy.groupby('patient_id').agg(
    total_patches=('filepath', 'count'),
    true_cancer_patches=('true_patch_label', 'sum'),
    pred_cancer_patches=('pred_patch_label', 'sum')
).reset_index()

# True Patient Diagnosis: If there is at least 1 actual cancer patch in their folder
patient_results['true_patient_label'] = (patient_results['true_cancer_patches'] > 0).astype(int)

# Predicted Patient Diagnosis: 2% threshold for extreme clinical sensitivity
threshold_percentage = 0.02
patient_results['pred_patient_label'] = (
    patient_results['pred_cancer_patches'] / patient_results['total_patches'] >= threshold_percentage
).astype(int)

# Calculate Patient-Level Confusion Matrix (forcing labels [0, 1] to avoid crashes)
patient_cm = confusion_matrix(patient_results['true_patient_label'], patient_results['pred_patient_label'], labels=[0, 1])
tn, fp, fn, tp = patient_cm.ravel()

# Final Metrics
print("--- PATIENT-LEVEL METRICS ---")
print(f"Total Patients Analyzed: {len(patient_results)}")
print(f"True Positives (Patients correctly diagnosed with cancer): {tp}")
print(f"True Negatives (Patients correctly diagnosed as healthy): {tn}")
print(f"False Positives (Healthy patients falsely alarmed): {fp}")
print(f"False Negatives (Cancer patients sent home - CRITICAL ERROR): {fn}\n")

print("--- PATIENT-LEVEL CLASSIFICATION REPORT ---")
print(classification_report(patient_results['true_patient_label'], patient_results['pred_patient_label'], labels=[0, 1], target_names=['Healthy Patient (0)', 'Cancerous Patient (1)']))

# Plot Patient-Level Confusion Matrix (Using an Orange theme to distinguish from patch-level)
plt.figure(figsize=(8, 6))
sns.heatmap(patient_cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Healthy Patient (0)', 'Cancerous Patient (1)'],
            yticklabels=['Healthy Patient (0)', 'Cancerous Patient (1)'],
            annot_kws={"size": 16})

plt.title('Patient-Level Confusion Matrix (Weighted Model)', fontsize=16)
plt.ylabel('True Patient Diagnosis', fontsize=14)
plt.xlabel('Predicted Patient Diagnosis', fontsize=14)
plt.show()